# 📐 Notebook 03 — Statistiques pour la Data Science

Les statistiques permettent de passer de l'**exploration** à la **compréhension** des données.

### 🎯 Objectifs de ce notebook
- Calculer et interpréter les statistiques descriptives
- Comprendre les distributions
- Mesurer les corrélations
- Réaliser des tests statistiques simples
- Appliquer la loi des grands nombres et le TLC

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)

titanic = sns.load_dataset('titanic')
tips    = sns.load_dataset('tips')
iris    = sns.load_dataset('iris')

print("✅ Prêt !")

## 1. Statistiques descriptives

### 1.1 Mesures de tendance centrale

In [ ]:
ages = titanic['age'].dropna()

print("=== TENDANCE CENTRALE ===")
print(f"  Moyenne   : {ages.mean():.1f} ans")
print(f"  Médiane   : {ages.median():.1f} ans")
print(f"  Mode      : {ages.mode()[0]:.1f} ans")

print("\n=== DISPERSION ===")
print(f"  Écart-type : {ages.std():.1f}")
print(f"  Variance   : {ages.var():.1f}")
print(f"  Min / Max  : {ages.min():.0f} / {ages.max():.0f}")
print(f"  IQR (Q3-Q1): {ages.quantile(0.75) - ages.quantile(0.25):.1f}")

print("\n=== QUARTILES ===")
for q in [0.25, 0.5, 0.75, 0.9]:
    print(f"  Q{int(q*100):3d}% : {ages.quantile(q):.1f} ans")

In [ ]:
# Visualisation : Moyenne vs Médiane
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Distribution des âges
axes[0].hist(ages, bins=30, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].axvline(ages.mean(), color='red', linestyle='--', linewidth=2, label=f'Moyenne ({ages.mean():.1f})')
axes[0].axvline(ages.median(), color='orange', linestyle='-', linewidth=2, label=f'Médiane ({ages.median():.1f})')
axes[0].set_title('Distribution des âges (Titanic)')
axes[0].legend()

# Boxplot
axes[1].boxplot(ages, vert=True, patch_artist=True,
                boxprops=dict(facecolor='steelblue', alpha=0.7))
axes[1].set_title('Boxplot des âges')
axes[1].set_ylabel('Âge')

plt.tight_layout()
plt.show()

## 2. Distributions statistiques

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

x = np.linspace(-4, 4, 200)

# Distribution normale
for mu, sigma, label in [(0, 1, 'μ=0, σ=1'), (0, 2, 'μ=0, σ=2'), (1, 1, 'μ=1, σ=1')]:
    axes[0].plot(x, stats.norm.pdf(x, mu, sigma), linewidth=2, label=label)
axes[0].set_title('Distribution Normale (Gaussienne)')
axes[0].legend(fontsize=9)

# Distribution uniforme
data_uniform = np.random.uniform(0, 10, 1000)
axes[1].hist(data_uniform, bins=20, color='coral', edgecolor='white', density=True)
axes[1].set_title('Distribution Uniforme')

# Distribution asymétrique
data_skewed = np.random.exponential(scale=2, size=1000)
axes[2].hist(data_skewed, bins=30, color='mediumpurple', edgecolor='white', density=True)
axes[2].set_title('Distribution Asymétrique (Exponentielle)')

plt.suptitle('Principales distributions statistiques', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Asymétrie (skewness) et aplatissement (kurtosis)
print("=== FORME DE LA DISTRIBUTION ===")
print(f"\nÂge des passagers Titanic :")
print(f"  Asymétrie (skewness) : {ages.skew():.3f}")
print(f"  → {'Distribution asymétrique à droite' if ages.skew() > 0 else 'Distribution asymétrique à gauche' if ages.skew() < 0 else 'Distribution symétrique'}")
print(f"  Aplatissement (kurtosis) : {ages.kurtosis():.3f}")

print(f"\nPourboires (tips) :")
print(f"  Asymétrie : {tips['tip'].skew():.3f}")
print(f"  Aplatissement : {tips['tip'].kurtosis():.3f}")

## 3. Corrélations

In [ ]:
# Matrice de corrélation — iris
iris_num = iris.drop('species', axis=1)
corr = iris_num.corr()

print("=== MATRICE DE CORRÉLATION (Iris) ===")
print(corr.round(3))

In [ ]:
# Visualisation
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Heatmap
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn',
            square=True, ax=axes[0], vmin=-1, vmax=1, center=0)
axes[0].set_title('Heatmap de corrélation')

# Scatter avec droite de régression
axes[1].scatter(iris['petal_length'], iris['petal_width'],
                c=pd.Categorical(iris['species']).codes,
                cmap='viridis', alpha=0.6)
# Droite de régression
slope, intercept, r, p, se = stats.linregress(iris['petal_length'], iris['petal_width'])
x_line = np.linspace(iris['petal_length'].min(), iris['petal_length'].max(), 100)
axes[1].plot(x_line, slope * x_line + intercept, 'r-', linewidth=2, label=f'r = {r:.3f}')
axes[1].set_xlabel('Longueur pétale')
axes[1].set_ylabel('Largeur pétale')
axes[1].set_title('Corrélation : longueur vs largeur pétale')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"\n📊 Corrélation de Pearson : r = {r:.3f} — {'Forte' if abs(r) > 0.7 else 'Modérée' if abs(r) > 0.3 else 'Faible'} corrélation")
print(f"   R² = {r**2:.3f} — La longueur explique {r**2*100:.1f}% de la variance de la largeur")

## 4. Tests statistiques

In [ ]:
# Test t de Student — les survivants étaient-ils plus jeunes ?
survivants    = titanic[titanic['survived'] == 1]['age'].dropna()
non_survivants = titanic[titanic['survived'] == 0]['age'].dropna()

print("=== TEST T DE STUDENT ===")
print(f"  Âge moyen survivants     : {survivants.mean():.1f} ans")
print(f"  Âge moyen non-survivants : {non_survivants.mean():.1f} ans")

t_stat, p_value = stats.ttest_ind(survivants, non_survivants)
print(f"\n  Statistique t : {t_stat:.3f}")
print(f"  p-value       : {p_value:.4f}")

alpha = 0.05
if p_value < alpha:
    print(f"\n  ✅ p < {alpha} → Différence STATISTIQUEMENT SIGNIFICATIVE")
    print(f"     On rejette H₀ : les âges des deux groupes sont différents")
else:
    print(f"\n  ❌ p >= {alpha} → Pas de différence significative")

In [ ]:
# Test de normalité (Shapiro-Wilk)
print("=== TEST DE NORMALITÉ (Shapiro-Wilk) ===")

for nom, donnees in [('Âge Titanic', ages[:50]), ('Pourboires', tips['tip'][:50])]:
    stat, p = stats.shapiro(donnees)
    print(f"\n  {nom}:")
    print(f"    Statistique W : {stat:.4f}")
    print(f"    p-value       : {p:.4f}")
    print(f"    Distribution  : {'Normale ✅' if p > 0.05 else 'Non-normale ❌'}")

In [ ]:
# Test du Chi-2 — indépendance entre deux variables catégorielles
print("=== TEST DU CHI-2 ===")
print("Question : La survie dépend-elle du sexe ?")

tableau = pd.crosstab(titanic['sex'], titanic['survived'])
print(f"\nTableau de contingence :")
print(tableau)

chi2, p, dof, expected = stats.chi2_contingency(tableau)
print(f"\n  Chi2      : {chi2:.2f}")
print(f"  p-value   : {p:.6f}")
print(f"  → {'Lien significatif ✅ — La survie dépend du sexe' if p < 0.05 else 'Pas de lien significatif'}")

## 5. Le Théorème Central Limite (TCL)

In [ ]:
# Démonstration du TCL
population = np.random.exponential(scale=2, size=10000)  # Distribution non-normale

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

tailles = [1, 5, 30, 100]

for ax, n in zip(axes, tailles):
    moyennes = [np.mean(np.random.choice(population, n)) for _ in range(1000)]
    ax.hist(moyennes, bins=30, color='steelblue', edgecolor='white', density=True)
    ax.set_title(f'n = {n}\nMoy.={np.mean(moyennes):.2f}, σ={np.std(moyennes):.2f}')

plt.suptitle('Théorème Central Limite — Distribution des moyennes échantillonnales', fontsize=12)
plt.tight_layout()
plt.show()

print("💡 Plus n est grand, plus la distribution des moyennes se rapproche d'une loi normale !")

## 6. ✏️ Exercices pratiques

In [ ]:
# EXERCICE 1
# Avec le dataset 'tips' :
# - Calculez les statistiques descriptives du pourboire ('tip') par jour
# - Y a-t-il une différence significative entre le vendredi et le samedi ? (test t)

# Votre code ici


In [ ]:
# EXERCICE 2
# Avec le dataset 'iris' :
# - Calculez la corrélation entre toutes les variables numériques
# - Quelle paire de variables est la plus corrélée ?
# - Créez un graphique qui illustre cette corrélation

# Votre code ici


---

## ✅ Résumé du notebook 03

| Concept | Ce que vous avez appris |
|---------|-------------------------|
| Descriptives | Moyenne, médiane, écart-type, quartiles |
| Distributions | Normale, uniforme, asymétrique, TCL |
| Corrélations | Pearson, R², matrice de corrélation |
| Tests statistiques | Test t, Chi-2, Shapiro-Wilk, p-value |

## ➡️ Prochain notebook : [04 — Machine Learning](04_machine_learning.ipynb)

---
*Formation Data Science Python — Notebook 03/04*